In [2]:
import pandas as pd
from requests import get
from bs4 import BeautifulSoup as bs
import time

In [3]:
# create an empty dataframe
df = pd.DataFrame()

# loop over pages indexes
for index in range(1, 21):

    url = f"https://www.gutenberg.org/ebooks/bookshelf/639?start_index={(index-1)*15}"
    res = get(url)
    soup = bs(res.content, "html.parser")

    containers = soup.find_all("li", class_="booklink")

    data = []

    for container in containers:
        try:
            # title
            title = container.find("span", class_="title").text.strip()

            # author
            author_tag = container.find("span", class_="subtitle")
            author = author_tag.text.strip() if author_tag else None

            # book URL
            link_tag = container.find("a")
            book_url = "https://www.gutenberg.org" + link_tag.get("href")

            # ebook number (from URL)
            ebook_no = link_tag.get("href").split("/")[-1]

            # go inside book page to get more details
            res_book = get(book_url)
            soup_book = bs(res_book.content, "html.parser")

            # release date
            release_date = None
            downloads = None

            table = soup_book.find("table", class_="bibrec")
            rows = table.find_all("tr") if table else []

            for row in rows:
                header = row.find("th")
                value = row.find("td")

                if header and value:
                    if "Release Date" in header.text:
                        release_date = value.text.strip()
                    if "Downloads" in header.text:
                        downloads = value.text.strip()

            dic = {
                "title": title,
                "author": author,
                "release_date": release_date,
                "book_url": book_url,
                "ebook_no": ebook_no,
                "downloads": downloads
            }

            data.append(dic)

            time.sleep(0.5)

        except Exception as e:
            pass

    DF = pd.DataFrame(data)
    df = pd.concat([df, DF], axis=0).reset_index(drop=True)

    time.sleep(1) 

# save to CSV
df.to_csv("data/gutenberg_books.csv", index=False)

print("Scraping complete. Saved to gutenberg_books.csv")

Scraping complete. Saved to gutenberg_books.csv


In [4]:
df = pd.read_csv("data/gutenberg_books.csv")

df. head()

,title,author,release_date,book_url,ebook_no,downloads
0,Pride and Prejudice,Jane Austen,"Jun 1, 1998",https://www.gutenberg.org/ebooks/1342,1342,108204 downloads in the last 30 days.
1,Romeo and Juliet,William Shakespeare,"Nov 1, 1998",https://www.gutenberg.org/ebooks/1513,1513,80862 downloads in the last 30 days.
2,A Room with a View,E. M. Forster,"May 1, 2001",https://www.gutenberg.org/ebooks/2641,2641,69058 downloads in the last 30 days.
3,The Blue Castle: a novel,L. M. Montgomery,"May 3, 2022",https://www.gutenberg.org/ebooks/67979,67979,55921 downloads in the last 30 days.
4,Jane Eyre: An Autobiography,Charlotte Brontë,"Mar 1, 1998",https://www.gutenberg.org/ebooks/1260,1260,54515 downloads in the last 30 days.


In [5]:
df.shape

(500, 6)

In [6]:
df.isnull().sum()

title            0
author          13
release_date     0
book_url         0
ebook_no         0
downloads        0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(191)

In [ ]:
df = df.drop_duplicates()